In [3]:
import os
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import xarray as xr
import matplotlib
matplotlib.use("Agg")  # don't display, just save
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import cartopy.crs as ccrs
import geopandas as gpd

# ---------------- Paths (same as before) ----------------
NC_PATH  = r"C:\Drought\Regridding and data clipping\VDI_v02_ESI\VDI_with_ESI_monthly_India_0p05deg.nc"
SHP_PATH = r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp"
OUT_DIR  = r"C:\Drought\Figure"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Load data (no dask) -------------------
print("Opening dataset…")
ds = xr.open_dataset(NC_PATH)  # default (in-memory ops)

# Variables (exclude uncertainty)
var_names = [v for v in ds.data_vars if v != "VDI_uncert"]

# Shapefile for border overlay
print("Loading India shapefile…")
india = gpd.read_file(SHP_PATH)
try:
    india = india.to_crs(epsg=4326)
except Exception:
    pass

# Cartopy projection
proj = ccrs.PlateCarree()

# ---------------- Helpers -------------------------------
def monthly_to_season_yearly(da: xr.DataArray, season: str) -> xr.DataArray:
    """
    Return seasonal (or annual) means with a 'year' dimension.
    Seasons: 'Annual', 'DJF', 'MAM', 'JJAS', 'ON'
    """
    m = da.time.dt.month
    y = da.time.dt.year
    if season == "Annual":
        return da.groupby(y.rename("year")).mean("time")
    if season == "DJF":
        # December goes to the next DJF year
        season_year = xr.where(m == 12, y + 1, y).rename("year")
        return da.where(m.isin([12, 1, 2])).groupby(season_year).mean("time")
    if season == "MAM":
        return da.where(m.isin([3, 4, 5])).groupby(y.rename("year")).mean("time")
    if season == "JJAS":
        return da.where(m.isin([6, 7, 8, 9])).groupby(y.rename("year")).mean("time")
    if season == "ON":
        return da.where(m.isin([10, 11])).groupby(y.rename("year")).mean("time")
    raise ValueError("Season must be one of: Annual, DJF, MAM, JJAS, ON")

def nice_limits_numpy(arr: np.ndarray, q=2.5, symmetric=True):
    """Percentile-based limits using NumPy (robust to NaNs)."""
    lo = np.nanpercentile(arr, q)
    hi = np.nanpercentile(arr, 100 - q)
    if symmetric:
        m = max(abs(lo), abs(hi))
        return -m, m
    return lo, hi

def plot_map(ax, da2d: xr.DataArray, title: str, vmin: float, vmax: float, cmap="RdBu_r"):
    """Generic spatial plot with India border overlay."""
    ax.set_extent([float(ds.lon.min()), float(ds.lon.max()),
                   float(ds.lat.min()), float(ds.lat.max())], crs=proj)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    im = da2d.plot.pcolormesh(ax=ax, transform=proj, cmap=cmap, norm=norm,
                              add_colorbar=False, rasterized=True)
    # overlay border
    india.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black", zorder=3)
    ax.coastlines(resolution="10m", linewidth=0.4)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    return im

# Consistent figure text sizes
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

# Seasons we use
SEASONS_ALL = ["Annual", "DJF", "MAM", "JJAS", "ON"]
SEASONS_YEARLY = ["DJF", "MAM", "JJAS", "ON"]  # yearly maps for these 4 seasons

# ========================================================
# 1) Spatial MEAN distributions (Annual + DJF + MAM + JJAS + ON)
# ========================================================
print("Making spatial mean (climatology) maps…")
for v in var_names:
    # Compute seasonal/annual mean maps (mean across years)
    maps = {}
    for s in SEASONS_ALL:
        seas = monthly_to_season_yearly(ds[v], s)  # (year, lat, lon)
        maps[s] = seas.mean("year")                # (lat, lon)

    # Color limits consistent across the 5 panels (use NumPy percentiles)
    stack = np.stack([maps[s].values for s in SEASONS_ALL], axis=0)  # shape: 5 x lat x lon
    vmin, vmax = nice_limits_numpy(stack, q=2.5, symmetric=True)

    # Panel layout: 2x3 (Annual, DJF, MAM, JJAS, ON) + one empty
    fig = plt.figure(figsize=(12, 6))
    gs = fig.add_gridspec(2, 3)
    axes = [
        fig.add_subplot(gs[0, 0], projection=proj),  # Annual
        fig.add_subplot(gs[0, 1], projection=proj),  # DJF
        fig.add_subplot(gs[0, 2], projection=proj),  # MAM
        fig.add_subplot(gs[1, 0], projection=proj),  # JJAS
        fig.add_subplot(gs[1, 1], projection=proj),  # ON
    ]
    order = ["Annual", "DJF", "MAM", "JJAS", "ON"]
    last_im = None
    for ax, s in zip(axes, order):
        last_im = plot_map(ax, maps[s], f"{v} — {s} mean", vmin, vmax)

    # Empty panel
    ax_empty = fig.add_subplot(gs[1, 2])
    ax_empty.axis("off")

    # Shared colorbar
    cax = fig.add_axes([0.25, 0.05, 0.5, 0.02])
    cbar = fig.colorbar(last_im, cax=cax, orientation="horizontal")
    cbar.set_label("Mean over available period")

    out = os.path.join(OUT_DIR, f"{v}_Climatology_Annual_DJF_MAM_JJAS_ON.png")
    plt.tight_layout(rect=[0, 0.07, 1, 0.98])
    plt.savefig(out, dpi=300)
    plt.close(fig)
    print("Saved:", out)

# ========================================================
# 2) For each season, every-year distribution maps in one figure
#    (for DJF, MAM, JJAS, ON) — for ALL variables
# ========================================================
print("Making per-season yearly maps…")
for v in var_names:
    for s in SEASONS_YEARLY:
        seas = monthly_to_season_yearly(ds[v], s)  # (year, lat, lon)
        # compute robust symmetric color limits across ALL years in this season
        vmin, vmax = nice_limits_numpy(seas.values, q=2.5, symmetric=True)
        years = seas["year"].values
        nY = len(years)

        # panel grid
        cols = 6  # compact layout
        rows = int(np.ceil(nY / cols))
        fig = plt.figure(figsize=(cols * 2.1, rows * 2.1))
        axes = [fig.add_subplot(rows, cols, i + 1, projection=proj) for i in range(rows * cols)]
        last_im = None

        for ax, yr in zip(axes, years):
            m = seas.sel(year=yr)
            last_im = plot_map(ax, m, f"{int(yr)}", vmin, vmax)

        # hide unused axes
        for ax in axes[len(years):]:
            ax.axis("off")

        fig.suptitle(f"{v} — {s} maps by year", y=0.92, fontsize=11)
        if last_im is not None:
            cax = fig.add_axes([0.25, 0.06, 0.5, 0.02])
            cbar = fig.colorbar(last_im, cax=cax, orientation="horizontal")
            cbar.set_label("Value")

        out = os.path.join(OUT_DIR, f"{v}_{s}_Yearly_Maps.png")
        plt.tight_layout(rect=[0, 0.08, 1, 0.9])
        plt.savefig(out, dpi=300)
        plt.close(fig)
        print("Saved:", out)

print("\nAll figures were saved to:", OUT_DIR)


<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_24832\1763836371.py:2: SyntaxWarning: invalid escape sequence '\D'
  """


Opening dataset…
Loading India shapefile…
Making spatial mean (climatology) maps…
Saved: C:\Drought\Figure\TVDI_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\VOD_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\NDVI_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\SIF_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\SMsurf_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\SMroot_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\ESI_dryz_Climatology_Annual_DJF_MAM_JJAS_ON.png
Saved: C:\Drought\Figure\VDI_Climatology_Annual_DJF_MAM_JJAS_ON.png
Making per-season yearly maps…
Saved: C:\Drought\Figure\TVDI_dryz_DJF_Yearly_Maps.png
Saved: C:\Drought\Figure\TVDI_dryz_MAM_Yearly_Maps.png
Saved: C:\Drought\Figure\TVDI_dryz_JJAS_Yearly_Maps.png
Saved: C:\Drought\Figure\TVDI_dryz_ON_Yearly_Maps.png
Saved: C:\Drought\Figure\VOD_dryz_DJF_Yearly_Maps.png
Saved: C:\Drought\Figure\V